# AI hubs and local compute: price, capacity and the adoption the corpus records

**Claim cluster:** `C081`, `C083`, `C084`, `C095`, `C096`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in these notebooks reads only from data/smarthome.duckdb.
CLAIM_IDS = ('C081', 'C083', 'C084', 'C095', 'C096')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

Nothing in this cluster is a documented fact: `C083`, `C084` and `C095` are reported signals at medium confidence, while `C081` and `C096` are inferences (`C096` at low confidence, from the trade press). The product figures behind them are recorded at low confidence in the metric table, and the panels make that visible before any bar is drawn.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### Price is the only attribute three of the four products share

`M011` prices the (rumoured) Apple Home Hub at 349-400 USD, `M015` prices Home Assistant Green at 199 USD and `M016` prices MasterAgent MA100 at 20,000 USD. Compute is recorded for only two products (`M013`, `M016`), so a price-versus-compute scatter would contain a single complete point and is not drawn. The price axis is logarithmic because the recorded range spans two orders of magnitude, and Anker MindBase is listed as 'no price recorded' rather than drawn at zero.

In [ ]:
# Price is written three different ways in this section, so each row states how
# its number was read:
#   M011  one range in one cell ("349-400")  -> viz.range_value
#   M015  one number in one cell ("199")     -> viz.range_value
#   M016  compute and price together ("2,070 TOPS; 20,000" with unit
#         "TOPS; USD")                       -> positional pairing by unit token
# The positional pairing is safe because the unit cell lists its units in the
# same order as the value cell lists its numbers, and it is stated here rather
# than left implicit.
hub_metrics = viz.metrics_by_id(["M011", "M013", "M015", "M016"])
hub_by_id = hub_metrics.set_index("local_id")


def _number_for_unit(metric_id, unit_token):
    """Number whose position matches the unit token in the metric's unit cell."""
    record = hub_by_id.loc[metric_id]
    numbers = viz.parse_numbers(record["value"])
    units = [token.strip() for token in str(record["unit"] or "").split(";")]
    if unit_token in units and units.index(unit_token) < len(numbers):
        return numbers[units.index(unit_token)]
    return None


_price_rows = []
for _mid, _product in (("M011", "Apple Home Hub (J490)"),
                       ("M015", "Home Assistant Green"),
                       ("M016", "MasterAgent MA100")):
    _low, _high = viz.range_value(hub_by_id.loc[_mid, "value"])
    if _mid == "M016":
        # range_value refuses a cell that holds two quantities, so the price is
        # taken by its unit token instead.
        _price = _number_for_unit("M016", "USD")
        _low, _high = _price, _price
        _reading = "price paired by unit token (cell holds compute and price)"
    else:
        _reading = "stated range" if _low != _high else "single stated value"
    _price_rows.append({
        "metric": _mid, "product": _product,
        "stated": hub_by_id.loc[_mid, "value"],
        "price_low": _low, "price_high": _high,
        "unit": hub_by_id.loc[_mid, "unit"],
        "confidence": hub_by_id.loc[_mid, "confidence"],
        "reading": _reading,
    })
priced = pd.DataFrame(_price_rows)
priced["compute_recorded"] = [
    "not recorded", "not recorded",
    hub_by_id.loc["M016", "value"] if "M016" in hub_by_id.index
    else "not recorded",
]
show(priced)

# The fourth product has compute but no recorded price, which is why it cannot
# appear on the price axis even though it is described in the same section.
if "M013" in hub_by_id.index:
    _anker = hub_by_id.loc["M013"]
    show(pd.DataFrame([{
        "metric": "M013", "product": "Anker MindBase",
        "stated": _anker["value"], "unit": _anker["unit"],
        "note": "no price recorded in the corpus",
        "confidence": _anker["confidence"],
    }]))

fig = go.Figure(go.Bar(
    x=[(low + high) / 2 for low, high in
       zip(priced["price_low"], priced["price_high"])],
    y=priced["product"] + " (" + priced["metric"] + ")",
    orientation="h",
    marker_color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                  for c in priced["confidence"]],
    error_x=dict(type="data",
                 array=[(high - low) / 2 for low, high in
                        zip(priced["price_low"], priced["price_high"])],
                 arrayminus=[(high - low) / 2 for low, high in
                             zip(priced["price_low"], priced["price_high"])],
                 visible=True),
    customdata=list(zip(priced["stated"], priced["compute_recorded"],
                        priced["reading"], priced["confidence"])),
    hovertemplate=("%{y}<br>stated cell: %{customdata[0]}"
                   "<br>compute recorded: %{customdata[1]}"
                   "<br>price reading: %{customdata[2]}"
                   "<br>confidence: %{customdata[3]}<extra></extra>"),
))
fig.update_layout(
    title=("Recorded hub prices, log scale (M011, M015, M016). Anker MindBase "
           "has no recorded price"),
    xaxis=dict(title="USD, as stated", type="log"),
    height=340,
    yaxis=dict(automargin=True),
)
fig.show()

## Supporting context

### What the corpus records about the market around the hardware

`M017` records private-label and utility-bundle brands at 30-35% of smart-plug unit shipments and `M018` puts the top five specialist brands at about 40%; the corpus records no figure for the remainder, so the two recorded shares are shown without a third bar and without a normalised pie. `C095` and `C096` add the two platform shifts the corpus noticed: a consumer router shipping as a Matter bridge, and camera support described in the trade press as a catalyst. `M012` records a modelled subscription breakeven of 20 months for the Apple hub, which is a business-model figure rather than a product one.

In [ ]:
display(HTML(
    "<h4>Recorded shares of smart-plug unit shipments</h4>"
    "<p>Two shares are recorded; the corpus records no figure for the rest of "
    "the market, so no remainder is drawn.</p>"
))
_shares = viz.metrics_by_id(["M017", "M018"])
show(_shares[["local_id", "metric_name", "value", "unit", "confidence",
              "conditions_stated", "source_ref"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "conditions_stated": "cond_stated", "source_ref": "source",
             "confidence": "conf"}))

fig = go.Figure(go.Bar(
    x=[viz.numeric_value(value) for value in _shares["value"]],
    y=[f"{record['local_id']} {record['metric_name']}"
       for record in _shares.to_dict("records")],
    orientation="h",
    marker_color=[viz.CONFIDENCE_COLORS.get(c, viz.CONFIDENCE_COLORS[None])
                  for c in _shares["confidence"]],
    text=[record["value"] for record in _shares.to_dict("records")],
    textposition="outside",
    customdata=list(zip(_shares["value"], _shares["unit"],
                        _shares["conditions_stated"])),
    hovertemplate=("%{y}<br>stated: %{customdata[0]} %{customdata[1]}"
                   "<br>conditions stated: %{customdata[2]}<extra></extra>"),
))
fig.update_layout(
    title=("Recorded share of smart-plug shipments (M017, M018). The "
           "remainder is not recorded and is not drawn"),
    xaxis_title="percent of unit shipments",
    height=300,
    yaxis=dict(automargin=True, autorange="reversed"),
)
fig.show()

display(HTML(
    "<h4>Hub adoption and platform-shift claims</h4>"
    "<p><code>M008</code> (5% of US internet households) is the corpus's "
    "hub-adoption figure; <code>M010</code> (52%) and <code>M009</code> (19%) "
    "are the adjacent ownership and attach rates, drawn in wave 1's business "
    "notebook and echoed here only as context, each against its own "
    "denominator.</p>"
))
show(viz.metrics_by_id(["M008", "M010", "M009", "M012", "M014"])[
    ["local_id", "metric_name", "value", "unit", "confidence", "source_ref"]]
     .rename(columns={"local_id": "metric", "metric_name": "name",
                      "source_ref": "source", "confidence": "conf"}))
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### Prices are estimates, and compute is a vendor claim

`M011` is recorded at low confidence and the Apple Home Hub entity itself is typed `Product (rumoured)`; `M013`, `M014` and `M016` are vendor-stated capacity and compute figures. The chart above is therefore a picture of what the market was reported to be launching, not a price list.

Other limits on this page:

- **No unit shipments are recorded.** The corpus holds two shares (`M017`, `M018`) and no volumes, so nothing here can be scaled to a market size.
- **`M017` has no stated conditions** (`conditions_stated = false`), so the 30-35% figure carries a geography, a period and a measurement method that the corpus never recorded.
- **Compute figures are not comparable.** `M013` states 26 TOPS and `M016` states 2,070 TOPS for devices in different classes; they are kept in one table only because the corpus records both in the same section.
- **`C096` is a low-confidence inference** that camera support could act as a catalyst for Matter adoption. It is a trade-press reading, not a measurement.
- **`C095` has no counterpart in the metric table.** The router that acts as a Matter bridge is described in a claim; no metric records its share, price or reach.

In [ ]:
# Vendor-claim metrics are separated from survey metrics, because 'a vendor
# states' and 'a survey found' are different kinds of number.
_vendor_ids = ["M011", "M012", "M013", "M014", "M016", "M018"]
_survey_ids = ["M008", "M017"]
display(HTML(
    "<h4>Vendor-stated figures against survey or market figures</h4>"
    "<p>The split is by source, not by confidence level.</p>"
))
_provenance = viz.metrics_by_id(_vendor_ids + _survey_ids)
_provenance["kind"] = ["vendor or trade statement"] * len(_vendor_ids) + \
                      ["survey or market report"] * len(_survey_ids)
show(_provenance[["local_id", "metric_name", "kind", "value", "unit",
                  "confidence", "source_ref"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Prices and shares are read by explicit metric id and parsed with `viz.range_value` (ranges) and `viz.numeric_value` (single values). The survey rows (`M008`, `M009`, `M010`) are shown as context only; wave 1's business notebook charts them. No figure is transcribed from the vendor announcements in the corpus.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()